# Human-in-the-Loop AI Data Annotator
### Google Colab edition — chatbot responses + customer reviews

This notebook lets you:
1. Create/load starter datasets.
2. Annotate chatbot responses and customer reviews.
3. Track annotation completion and label distributions.
4. Export your labelled CSV files.

**How to use:** Open this notebook in Google Colab and run cells from top to bottom. The sample records are practice examples, not a verified benchmark dataset.

In [ ]:
# CELL 1 — Install/import libraries
!pip -q install pandas plotly ipywidgets

import os
import pandas as pd
import plotly.express as px
from IPython.display import display, clear_output
import ipywidgets as widgets

print("Setup complete!")


In [ ]:
# CELL 2 — Create starter datasets
os.makedirs("/content/ai_data_annotator", exist_ok=True)
DATA_DIR = "/content/ai_data_annotator"

chatbot_path = f"{DATA_DIR}/chatbot_responses.csv"
reviews_path = f"{DATA_DIR}/customer_reviews.csv"

chatbot_columns = [
    "record_id", "prompt", "response", "relevance", "factuality",
    "clarity", "safety", "bias", "quality_score", "annotator_comment"
]
review_columns = [
    "record_id", "review", "sentiment", "topic", "spam",
    "language_quality", "priority", "annotator_comment"
]

chatbot_examples = [
    ["C001","Explain what machine learning is.","Machine learning is a field of AI where systems learn patterns from data to make predictions or decisions.","","","","","","",""],
    ["C002","How can I reset my account password?","Open the sign-in page, choose 'Forgot password', and follow the reset link sent to your registered email.","","","","","","",""],
    ["C003","What is photosynthesis?","Photosynthesis is the process by which plants use light energy to convert carbon dioxide and water into sugars, releasing oxygen.","","","","","","",""],
    ["C004","Give me a safe way to store passwords.","Use a reputable password manager and enable multi-factor authentication. Avoid reusing passwords.","","","","","","",""],
    ["C005","What is the capital of France?","The capital of France is Berlin.","","","","","","",""],
    ["C006","Summarize this product return policy.","I cannot determine the policy without seeing the actual policy text. Please share it and I can summarize it.","","","","","","",""],
    ["C007","How do I improve my writing?","Read regularly, write consistently, revise for clarity, and ask for feedback from trusted readers.","","","","","","",""],
    ["C008","Can you guarantee this investment will double?","Yes, every investment is guaranteed to double within one year.","","","","","","",""],
    ["C009","Explain two-factor authentication simply.","It adds a second verification step, such as an authenticator code, in addition to your password.","","","","","","",""],
    ["C010","Tell me about renewable energy.","Renewable energy comes from sources replenished naturally, such as sunlight, wind, and flowing water.","","","","","","",""]
]

review_examples = [
    ["R001","The headphones sound excellent and the battery lasts all day.","","","","","",""],
    ["R002","Delivery arrived four days late and the box was damaged.","","","","","",""],
    ["R003","Good value for the price. Does exactly what I need.","","","","","",""],
    ["R004","Customer support never replied to my emails.","","","","","",""],
    ["R005","The app is easy to use, but it crashes sometimes.","","","","","",""],
    ["R006","Amazing amazing amazing buy now click my link!!!","","","","","",""],
    ["R007","The fabric feels comfortable and the stitching is neat.","","","","","",""],
    ["R008","It is okay. Nothing special, but not bad either.","","","","","",""],
    ["R009","The item is much smaller than the product description suggested.","","","","","",""],
    ["R010","Fast shipping, helpful support, and a smooth return process.","","","","","",""]
]

if not os.path.exists(chatbot_path):
    pd.DataFrame(chatbot_examples, columns=chatbot_columns).to_csv(chatbot_path, index=False)
if not os.path.exists(reviews_path):
    pd.DataFrame(review_examples, columns=review_columns).to_csv(reviews_path, index=False)

chatbot_df = pd.read_csv(chatbot_path).fillna("")
reviews_df = pd.read_csv(reviews_path).fillna("")

print("Chatbot records:", len(chatbot_df))
print("Customer review records:", len(reviews_df))
display(chatbot_df.head(3))
display(reviews_df.head(3))


In [ ]:
# CELL 3 — Optional: upload your own CSV files
# Upload files with these names to replace the starter datasets:
# chatbot_responses.csv and/or customer_reviews.csv

from google.colab import files

print("Optional: choose your own CSV files. Cancel/close the picker to keep sample data.")
uploaded = files.upload()

for filename, content in uploaded.items():
    if filename.lower() in ["chatbot_responses.csv", "customer_reviews.csv"]:
        destination = f"{DATA_DIR}/{filename.lower()}"
        with open(destination, "wb") as f:
            f.write(content)
        print("Loaded:", filename)

chatbot_df = pd.read_csv(chatbot_path).fillna("")
reviews_df = pd.read_csv(reviews_path).fillna("")


## Annotation rules

- **Relevance:** Relevant / Partially relevant / Irrelevant / Needs review.
- **Factuality:** Supported / Partially supported / Unsupported / Needs review. Verify claims against reliable sources; don't judge only by how confident the response sounds.
- **Clarity:** Clear / Somewhat unclear / Unclear.
- **Safety:** Safe / Potentially harmful / Needs review.
- **Bias:** No obvious bias / Potential bias / Needs review.
- **Review sentiment:** Positive / Neutral / Negative / Mixed.
- **Review topic:** Product quality / Delivery / Price / Customer service / Usability / Other.
- **Spam:** Not spam / Possible spam / Spam.

Use **Needs review** when the available evidence is insufficient. For a genuine annotator-agreement study, ask a second person to independently label the same records.

In [ ]:
# CELL 4 — Interactive annotation tool
chat_options = {
    "relevance": ["Relevant", "Partially relevant", "Irrelevant", "Needs review"],
    "factuality": ["Supported", "Partially supported", "Unsupported", "Needs review"],
    "clarity": ["Clear", "Somewhat unclear", "Unclear"],
    "safety": ["Safe", "Potentially harmful", "Needs review"],
    "bias": ["No obvious bias", "Potential bias", "Needs review"],
    "quality_score": ["1", "2", "3", "4", "5"]
}
review_options = {
    "sentiment": ["Positive", "Neutral", "Negative", "Mixed"],
    "topic": ["Product quality", "Delivery", "Price", "Customer service", "Usability", "Other"],
    "spam": ["Not spam", "Possible spam", "Spam"],
    "language_quality": ["Understandable", "Difficult to understand"],
    "priority": ["Low", "Medium", "High"]
}

dataset_widget = widgets.Dropdown(
    options=["Chatbot responses", "Customer reviews"],
    description="Dataset:",
    layout=widgets.Layout(width="350px")
)
record_widget = widgets.Dropdown(description="Record:", layout=widgets.Layout(width="350px"))
form_output = widgets.Output()
field_widgets = {}
comment_widget = widgets.Textarea(
    description="Comment:",
    placeholder="Add a short reason, especially for uncertain labels",
    layout=widgets.Layout(width="95%", height="80px")
)
save_button = widgets.Button(description="Save annotation", button_style="success", icon="save")

def current_df():
    return chatbot_df if dataset_widget.value == "Chatbot responses" else reviews_df

def refresh_records(*args):
    df = current_df()
    record_widget.options = list(df["record_id"])
    render_form()

def render_form(*args):
    df = current_df()
    if len(df) == 0:
        return
    row = df[df.record_id == record_widget.value].iloc[0]
    with form_output:
        clear_output(wait=True)
        if dataset_widget.value == "Chatbot responses":
            display(widgets.HTML(f"<h4>Prompt</h4><p>{row['prompt']}</p><h4>AI response</h4><p>{row['response']}</p>"))
            options = chat_options
            label_cols = list(options.keys())
        else:
            display(widgets.HTML(f"<h4>Customer review</h4><p>{row['review']}</p>"))
            options = review_options
            label_cols = list(options.keys())
        field_widgets.clear()
        for field, choices in options.items():
            current = str(row.get(field, "") or "")
            field_widgets[field] = widgets.Dropdown(
                options=[""] + choices,
                value=current if current in choices else "",
                description=field.replace("_", " ").title() + ":",
                layout=widgets.Layout(width="500px")
            )
            display(field_widgets[field])
        comment_widget.value = str(row.get("annotator_comment", "") or "")
        display(comment_widget)
        display(save_button)

def save_annotation(button):
    global chatbot_df, reviews_df
    df = current_df()
    idx = df.index[df.record_id == record_widget.value][0]
    for field, control in field_widgets.items():
        df.at[idx, field] = control.value
    df.at[idx, "annotator_comment"] = comment_widget.value
    if dataset_widget.value == "Chatbot responses":
        chatbot_df = df
        chatbot_df.to_csv(chatbot_path, index=False)
    else:
        reviews_df = df
        reviews_df.to_csv(reviews_path, index=False)
    with form_output:
        print(f"Saved annotation for {record_widget.value}")

dataset_widget.observe(refresh_records, names="value")
record_widget.observe(render_form, names="value")
save_button.on_click(save_annotation)

display(dataset_widget, record_widget, form_output)
refresh_records()


In [ ]:
# CELL 5 — Annotation progress and label dashboard
def completion_summary(df, label_columns):
    if len(df) == 0:
        return 0, 0
    completed = df[label_columns].fillna("").astype(str).apply(
        lambda row: all(value.strip() != "" for value in row), axis=1
    ).sum()
    return int(completed), len(df)

chat_completed, chat_total = completion_summary(
    chatbot_df, ["relevance", "factuality", "clarity", "safety", "bias", "quality_score"]
)
review_completed, review_total = completion_summary(
    reviews_df, ["sentiment", "topic", "spam", "language_quality", "priority"]
)

summary = pd.DataFrame([
    ["Chatbot responses", chat_total, chat_completed],
    ["Customer reviews", review_total, review_completed]
], columns=["Dataset", "Total records", "Fully annotated"])
summary["Completion %"] = (summary["Fully annotated"] / summary["Total records"] * 100).round(1)

display(summary)

metric_dataset = widgets.Dropdown(options=["Chatbot responses", "Customer reviews"], description="View:")
display(metric_dataset)

def show_distribution(change=None):
    df = chatbot_df if metric_dataset.value == "Chatbot responses" else reviews_df
    cols = chat_options if metric_dataset.value == "Chatbot responses" else review_options
    field = list(cols.keys())[0]
    counts = df[field].replace("", "Unlabelled").value_counts().rename_axis("Label").reset_index(name="Count")
    fig = px.bar(counts, x="Label", y="Count", title=f"{field.replace('_',' ').title()} distribution")
    fig.show()

metric_dataset.observe(show_distribution, names="value")
show_distribution()


In [ ]:
# CELL 6 — Export annotated CSV files
chatbot_df.to_csv(f"{DATA_DIR}/chatbot_annotations.csv", index=False)
reviews_df.to_csv(f"{DATA_DIR}/customer_review_annotations.csv", index=False)

print("Export files created:")
print(f"{DATA_DIR}/chatbot_annotations.csv")
print(f"{DATA_DIR}/customer_review_annotations.csv")

from google.colab import files
files.download(f"{DATA_DIR}/chatbot_annotations.csv")
files.download(f"{DATA_DIR}/customer_review_annotations.csv")


In [ ]:
# CELL 7 — Optional: download all project data as one ZIP
import shutil
zip_file = shutil.make_archive("/content/AI_Data_Annotator_Colab_Exports", "zip", DATA_DIR)
from google.colab import files
files.download(zip_file)


## Portfolio notes

Before publishing:
- Replace or expand the illustrative examples with a properly sourced dataset.
- Keep dataset source, licence, and collection date in your project documentation.
- Do not claim annotation accuracy unless you have trusted gold labels.
- Report completion rate separately from accuracy.
- For agreement, have another annotator label the same subset independently and calculate Cohen's kappa where appropriate.

**Suggested next improvement:** add a second independent annotator workflow and a confusion-matrix/quality-audit report.